# 0.45 — Grafo di esempio con lo schema 0.4

Un grafo costruito su 44 news Bloomberg reali del 2023, per vedere concretamente **nodi, archi e attributi** dello
schema 0.4 (definito in [0.44](0.44-ontology-extraction-and-temporal-leakage.ipynb#schema)) e come ne emerge un gruppo che
potrebbe essere un tema. Nodi e archi li ha estratti GPT-5.5 con il nostro estrattore.

**Indice**

1. [Le news](#news)
2. [Da una news a un piccolo grafo](#una-news)
3. [Il grafo completo: nodi, archi, attributi](#completo)
4. [Grafo interattivo (HTML)](#html)
5. [Come si vede un tema](#tema)
6. [Limiti di questo esempio](#limiti)
7. [Dati completi](#dati)

Il notebook legge estrazioni già salvate: gira in pochi secondi, senza chiamate API.

In [1]:
import json, sys, warnings
from collections import Counter
from pathlib import Path
import pandas as pd
import networkx as nx
from IPython.display import display, HTML

sys.path.insert(0, str(Path("../scripts").resolve()))
import extract_044 as X
import graph_045 as GR

X.RUN_API = False                                   # solo estrazioni salvate
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
pd.set_option("display.max_colwidth", 160)
news = [json.loads(l) for l in open("../data/processed/news_045_sample.jsonl")]
outs = [X.extract_v04(n["headline"], n["date"], "gpt-5.5", True) for n in news]
G = GR.build_graph(news, outs)
N = {n["id"]: n for n in news}
lab = lambda k: G.nodes[k]["label"]
print(f"{len(news)} news → {G.number_of_nodes()} nodi, {G.number_of_edges()} archi")

44 news → 96 nodi, 50 archi


<a id="news"></a>
## 1. Le news

44 news del 2023, in due gruppi scelti **con parole-chiave solo per avere un esempio leggibile**: intelligenza artificiale
e farmaci per l'obesità. Nella scoperta vera dei temi nessun nome di tema viene usato. Per gruppo: 16 news a caso, 3 con
parole da deal e 2 con parole da fornitura; due news in più sono scelte a mano per mostrare l'ultimo tipo di arco
(`SUPPLIES_TO`) e l'ultima modalità (negato). Campionatore: [sample_045.py](../scripts/sample_045.py).

In [2]:
display(pd.crosstab(pd.Series([n["group"] for n in news], name="gruppo"), pd.Series([n["how"] for n in news], name="come è stata scelta"),
                    margins=True, margins_name="totale"))

come è stata scelta,a caso,deal,scelta: SUPPLIES_TO,scelta: modalità negato,supply,totale
gruppo,,,,,,
altro,0,0,0,1,0,1
farmaci per l'obesità,16,3,0,0,2,21
intelligenza artificiale,16,3,1,0,2,22
totale,32,6,1,1,4,44


<a id="una-news"></a>
## 2. Da una news a un piccolo grafo

Ogni news diventa pochi nodi e pochi archi. Su ogni arco ci sono i quattro attributi: **data**, **modalità**, **evidenza**
(la frase copiata dalla news) e **news** (il suo id). Sette news scelte per mostrare tutti i tipi di arco e tutte le
modalità, più una news che produce solo nodi.

In [3]:
TYPE_IT = {k: v[0] for k, v in GR.NODE_STYLE.items()}
EDGE_IT = {k: v[0] for k, v in GR.EDGE_STYLE.items()}
MOD_IT = {k: v[0] for k, v in GR.MODALITY_STYLE.items()}

def show_news(fragment):
    i = next(k for k, n in enumerate(news) if fragment in n["headline"])
    n, o = news[i], outs[i]
    name = {x["id"]: x["mention"] for x in o["nodes"]}
    print(f"«{n['headline']}»  ({n['date']}, {n['id']})")
    print("   nodi :", " · ".join(f"{x['mention']} [{TYPE_IT[x['type']]}]" for x in o["nodes"]))
    for e in o["edges"]:
        print(f"   arco : {name[e['subject_id']]} ──{EDGE_IT[e['predicate']]}──▶ {name[e['object_id']]}"
              f"   {{data {n['date']} · modalità {MOD_IT[e['modality']]} · evidenza «{e['evidence']}»}}")
    if not o["edges"]:
        print("   archi: nessuno — la news cita i nodi ma non afferma una relazione")
    print()

for fragment in ["Roche to Buy Obesity Drug Maker Carmot", "Samsung Signs Deal With Nvidia", "Microsoft Adds $10 Billion",
                 "SoftBank and Symbotic", "Chipmakers Rally on Hopes", "Adobe and Figma Terminate", "Oracle Seeks Cloud AI",
                 "Pfizer Jumps After Journal"]:
    show_news(fragment)

«Roche to Buy Obesity Drug Maker Carmot for $3.1 Billion (2)»  (2023-12-04, n40)
   nodi : Roche [azienda] · Obesity Drug [prodotto] · Carmot [azienda]
   arco : Carmot ──attiva in──▶ Obesity Drug   {data 2023-12-04 · modalità fatto · evidenza «Obesity Drug Maker Carmot»}
   arco : Roche ──acquisisce──▶ Carmot   {data 2023-12-04 · modalità piano · evidenza «Roche to Buy Obesity Drug Maker Carmot»}

«Samsung Signs Deal With Nvidia to Supply HBM3: Daily»  (2023-09-01, n30)
   nodi : Samsung [azienda] · Nvidia [azienda] · HBM3 [prodotto] · Daily [azienda]
   arco : Samsung ──fornisce a──▶ Nvidia   {data 2023-09-01 · modalità fatto · evidenza «Samsung Signs Deal With Nvidia to Supply HBM3»}
   arco : Samsung ──offre──▶ HBM3   {data 2023-09-01 · modalità fatto · evidenza «Samsung Signs Deal With Nvidia to Supply HBM3»}

«Microsoft Adds $10 Billion to Investment in OpenAI: Daybreak»  (2023-01-23, n02)
   nodi : Microsoft [azienda] · OpenAI [azienda] · Daybreak [azienda]
   arco : Microsoft ─

<a id="completo"></a>
## 3. Il grafo completo: nodi, archi, attributi

Mettendo insieme le 44 news, lo stesso nodo citato in news diverse diventa un nodo solo (stesso tipo e stesso nome,
a meno di maiuscole, trattini e plurali). Due news che affermano lo stesso fatto danno due archi, ciascuno con la sua data.

In [4]:
def examples(keys, k=4):
    return ", ".join(sorted({lab(x) for x in keys}, key=str.lower)[:k])

node_rows = [{"tipo": f"{TYPE_IT[t]} ({t})", "nodi": sum(G.nodes[k]["type"] == t for k in G),
              "esempi": examples([k for k in G if G.nodes[k]["type"] == t])} for t in GR.NODE_STYLE]
edge_rows = []
for p in GR.EDGE_STYLE:
    es = [(s, t) for s, t, d in G.edges(data=True) if d["predicate"] == p]
    edge_rows.append({"tipo": f"{EDGE_IT[p]} ({p})", "archi": len(es),
                      "esempio": f"{lab(es[0][0])} → {lab(es[0][1])}" if es else "–"})
mod_rows = [{"modalità": f"{MOD_IT[m]} ({m})", "archi": sum(d["modality"] == m for *_, d in G.edges(data=True))} for m in GR.MODALITY_STYLE]
display(pd.DataFrame(node_rows).set_index("tipo"))
display(pd.DataFrame(edge_rows).set_index("tipo"))
display(pd.DataFrame(mod_rows).set_index("modalità"))
iso = [k for k in G if G.degree(k) == 0]
print(f"{len(iso)} nodi non hanno archi: le loro news li citano senza affermare relazioni (es. {examples(iso, 5)}).")

,nodi,esempi
tipo,,
azienda (COMPANY),46,"Adobe, ASMI, ASML, AstraZeneca"
prodotto (PRODUCT),21,"AI, AI Chip, Bariatric Surgery, Chatbot"
concetto (CONCEPT),13,"AI, AI Detection, AI Search, AI Systems"
settore (SECTOR),7,"BILLING, Chipmakers, Consumer Staples, Drugmaker"
organizzazione (ORGANIZATION),3,"EMA, Journal, Union"
luogo (LOCATION),6,"Asia, China, England, Japan"


,archi,esempio
tipo,,
offre (OFFERS),5,Meta → Tool
attiva in (ACTIVE_IN),21,Waabi → AI
esposta a (EXPOSED_TO),14,BigBear.ai → AI
investe in (INVESTS_IN),2,Volvo Group Venture Capital → Waabi
acquisisce (ACQUIRES),4,Novo Nordisk → Embark Biotech
partner di (PARTNERS_WITH),3,Bristol Myers Squibb → Viz.ai
fornisce a (SUPPLIES_TO),1,Samsung → Nvidia


,archi
modalità,
fatto (reported),37
piano (planned),5
incerto (uncertain),7
negato (negated),1


34 nodi non hanno archi: le loro news li citano senza affermare relazioni (es. AI Search, Asia, BofA, China, Daily).


<a id="html"></a>
## 4. Grafo interattivo (HTML)

La pagina è salvata in `code/notebooks/output/0.45_grafo_esempio.html`: si apre in qualunque browser. Cosa si può fare:

- **colori e forme** distinguono i tipi di nodo; il **colore della freccia** il tipo di arco; il **tratteggio** la modalità
  (continuo = fatto, tratteggiato = piano, puntinato = incerto, grigio = negato);
- **cursore della data** e pulsante ▶: il grafo cresce mese per mese, come crescerebbe un tema;
- **filtri** per tipo di nodo, tipo di arco e modalità; casella per mostrare anche i nodi senza archi;
- **clic** su un nodo o su una freccia: il pannello "Dettaglio" mostra gli attributi, l'evidenza e le news;
- **ricerca** per nome.

La dimensione di un nodo cresce con il numero di aziende collegate. In cima alla pagina, il riquadro "Come è stato
costruito" riporta estrattore (GPT-5.5), ontologia (schema 0.4) e filtro con cui sono state scelte le news. Se la pagina
qui sotto non si vede, aprire il file nel browser.

In [5]:
import html as _html, sample_045 as SM
served = outs[0]["_meta"]["served_by"]
kw = lambda pattern: ", ".join(f"<code>{_html.escape(w.replace(chr(92) + 'b', '').replace('[- ]', '-'))}</code>" for w in pattern.split("|"))
groups = " e ".join(f"<i>{g}</i> ({kw(p)})" for g, p in SM.GROUPS.items())
about = f"""
<p><b>Titoli:</b> {len(news)} news Bloomberg vere del 2023, dal corpus del progetto.</p>
<p><b>Filtro:</b> titoli che contengono parole-chiave di due gruppi: {groups}. Per gruppo {SM.N_RANDOM} a caso, più alcuni
con parole da deal o fornitura; {len(SM.EXTRA)} titoli scelti a mano (Samsung–Nvidia, Adobe–Figma) per mostrare ogni tipo di arco.</p>
<p><b>Estrattore:</b> GPT-5.5 (<code>{served}</code>, reasoning <code>none</code>) con la <b>nostra ontologia, schema 0.4</b>:
6 tipi di nodo, 7 tipi di arco, attributi data · modalità · evidenza · news (guideline 044-v2, notebook 0.44).
Nel prompt c'è la data della news. Nodi e archi non sono stati corretti a mano.</p>
<p class="warn"><b>Attenzione:</b> il nodo «AI» è così grande e centrale soprattutto perché i titoli sono stati <b>scelti
contenendo "AI"</b>: mostra come si forma un gruppo, non è una scoperta. Nodi uniti solo se hanno stesso tipo e stesso nome:
«AI», «Artificial Intelligence» e «AI Chip» restano separati. Ci sono errori di estrazione (es. «MEDICAL AI BILLING COMPANY»,
«C3.ai esposta a AI»).</p>"""
out = Path("output/0.45_grafo_esempio.html")
page = GR.to_html(G, news, "Grafo di esempio · schema 0.4", "44 news Bloomberg del 2023, estratte da GPT-5.5", about)
out.write_text(page)
print("Salvato:", out.resolve())
display(HTML(f'<iframe srcdoc="{page.replace("&", "&amp;").replace(chr(34), "&quot;")}" '
             'style="width:100%; height:760px; border:1px solid #ddd; border-radius:8px"></iframe>'))

Salvato: /Users/federicocinus/CODE - how/ThematicTrading/code/notebooks/output/0.45_grafo_esempio.html


<a id="tema"></a>
## 5. Come si vede un tema

Il tema non è un nodo: è un **gruppo** di prodotti e concetti a cui si collegano sempre più aziende. Qui un algoritmo
standard di comunità (greedy modularity di networkx, sul grafo senza direzioni) raggruppa i nodi collegati fra loro; per
ogni gruppo con almeno tre aziende contiamo le aziende collegate trimestre per trimestre. Con 44 news è solo
un'illustrazione: nel progetto si userà Leiden su finestre temporali, su milioni di news.

In [6]:
U = nx.Graph()
U.add_edges_from((s, t) for s, t in G.edges())
groups = sorted(nx.community.greedy_modularity_communities(U), key=len, reverse=True)
QUARTERS = [("T1", "2023-03-31"), ("T2", "2023-06-30"), ("T3", "2023-09-30"), ("T4", "2023-12-31")]
rows = []
for c in groups:
    comps = [k for k in c if G.nodes[k]["type"] == "COMPANY"]
    if len(comps) < 3:
        continue
    bricks = [k for k in c if G.nodes[k]["type"] in ("PRODUCT", "CONCEPT", "SECTOR")]
    edges_in = [d for s, t, d in G.edges(data=True) if s in c and t in c]
    row = {"mattoni (prodotti, concetti, settori)": examples(bricks, 8), "aziende": examples(comps, 8)}
    for q, end in QUARTERS:     # distinct companies with an edge inside the group, up to the end of the quarter
        row[f"aziende entro {q}"] = len({x for s, t, d in G.edges(data=True) if s in c and t in c and d["date"] <= end
                                         for x in (s, t) if G.nodes[x]["type"] == "COMPANY"})
    rows.append(row)
display(pd.DataFrame(rows))

,"mattoni (prodotti, concetti, settori)",aziende,aziende entro T1,aziende entro T2,aziende entro T3,aziende entro T4
0,"AI, HBM3","BigBear.ai, C3.ai, Microsoft, Nvidia, OpenAI, Samsung, Volvo Group Venture Capital, Waabi",5,7,8,8
1,"Diabetes Drug, Drugmaker, Mounjaro, Obesity Shot, Wegovy","Lilly, Viking, Weight Watchers",0,1,3,3
2,"Bariatric Surgery, Obesity Drugs","Carmot, Intuitive, Roche",0,0,1,3


Il gruppo dell'intelligenza artificiale si forma da solo attorno al nodo "AI", e le aziende collegate crescono nel corso
dell'anno: è il segnale che il rilevatore dei temi misurerà. Quello dei farmaci per l'obesità invece esce **spezzato**
(Lilly e Wegovy da una parte, Roche e "Obesity Drugs" dall'altra), perché senza un collegamento fra nomi diversi "Wegovy",
"Obesity Drugs" e "Weight-Loss Drugs" restano nodi separati: è esattamente il lavoro del vocabolario datato previsto dopo
l'estrazione. Il nome del tema non compare nel grafo: lo si dà alla fine, a un gruppo già trovato.

<a id="limiti"></a>
## 6. Limiti di questo esempio

- **Estrazione non verificata.** Per queste 44 news non c'è un gold: nodi e archi sono quelli di GPT-5.5, errori compresi.
  Se ne vedono alcuni: "MEDICAL AI BILLING COMPANY" è un'azienda senza nome e non dovrebbe essere un nodo; "Lilly attiva
  in Drugmaker" usa il nome del ruolo invece del settore; "Consumer Staples esposto a Rates" usa una variabile finanziaria;
  "JV" (joint venture) è preso per un'azienda.
- **Nessun collegamento fra nomi diversi della stessa cosa.** "AI", "Artificial Intelligence" e "AI Chip" sono nodi
  diversi, come "Novo" e "Novo Nordisk". Unirli richiede un vocabolario datato (passo successivo, point-in-time).
- **News scelte con parole-chiave** per avere due gruppi leggibili, e due news aggiunte a mano per mostrare ogni tipo di arco.

<a id="dati"></a>
## 7. Dati completi

Metti `MOSTRA = True` per vedere nodi e archi di tutte le 44 news.

In [7]:
MOSTRA = False

if MOSTRA:
    for n in news:
        show_news(n["headline"])
else:
    print("Nascosto. Variabili disponibili: news, outs (estrazioni), G (grafo networkx).")

Nascosto. Variabili disponibili: news, outs (estrazioni), G (grafo networkx).
